In [1]:
import pandas as pd
import numpy as np

#Facilitar a lectura de los numerosos valores
np.set_printoptions(precision=3, suppress=True)

import tensorflow as tf
from tensorflow.keras import layers
preprocessing = layers

In [2]:
air_accident = pd.read_csv("dataset/train.csv")
air_accident.head(10)

,survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,0,male,22.0,1,0,7.2500,Third,unknown,Southampton,n
1,1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
2,1,female,26.0,0,0,7.9250,Third,unknown,Southampton,y
3,1,female,35.0,1,0,53.1000,First,C,Southampton,n
4,0,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y
5,0,male,2.0,3,1,21.0750,Third,unknown,Southampton,n
6,1,female,27.0,0,2,11.1333,Third,unknown,Southampton,n
7,1,female,14.0,1,0,30.0708,Second,unknown,Cherbourg,n
8,1,female,4.0,1,1,16.7000,Third,G,Southampton,n
9,0,male,20.0,0,0,8.0500,Third,unknown,Southampton,y


In [3]:
air_accident_features = air_accident.copy()
air_accident_labels = air_accident_features.pop('survived')

creamos un modelo que implementa la lógica de preprocesamiento utilizando la API funcional de Keras.

In [4]:
#Creamos una entrada simbolica
input = tf.keras.Input(shape=(), dtype=tf.float32)
#Hacemos un calculo usando..
result = 2*input + 1
#Este resultado no va a tener valor
result

<KerasTensor shape=(None,), dtype=float32, sparse=False, ragged=False, name=keras_tensor_2>

construimos un conjunto de objetos keras.Input simbólicos, haciendo coincidir los nombres y tipos de datos de las columnas del CSV.

In [9]:
inputs = {}
for name, column in air_accident_features.items():
    if pd.api.types.is_numeric_dtype(column):
        dtype = tf.float32
    else:
        dtype = tf.string
    inputs[name] = tf.keras.Input(shape=(1,), name=name, dtype=dtype)
inputs

{'sex': <KerasTensor shape=(None, 1), dtype=string, sparse=False, ragged=False, name=sex>,
 'age': <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=age>,
 'n_siblings_spouses': <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=n_siblings_spouses>,
 'parch': <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=parch>,
 'fare': <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=fare>,
 'class': <KerasTensor shape=(None, 1), dtype=string, sparse=False, ragged=False, name=class>,
 'deck': <KerasTensor shape=(None, 1), dtype=string, sparse=False, ragged=False, name=deck>,
 'embark_town': <KerasTensor shape=(None, 1), dtype=string, sparse=False, ragged=False, name=embark_town>,
 'alone': <KerasTensor shape=(None, 1), dtype=string, sparse=False, ragged=False, name=alone>}

concatenamos las entradas numéricas juntas y ejecutamos a través de una capa de normalización

In [10]:
numeric_inputs = {name:input for name,input in inputs.items()
                  if input.dtype==tf.float32}
x = layers.Concatenate()(list(numeric_inputs.values()))
norm = preprocessing.Normalization()
norm.adapt(np.array(air_accident[list(numeric_inputs.keys())].astype(np.float32)))
all_numeric_inputs = norm(x)
all_numeric_inputs

<KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_6>

1. recopilamos todos los resultados del preprocesamiento simbólico, para concatenarlos posteriormente.
2. usamos la función preprocessing.StringLookup para mapear desde cadenas a índices enteros en un vocabulario.
3. concatenar todas las entradas preprocesadas juntas y construimos un modelo que maneja el preprocesamiento.


In [17]:
preprocessed_inputs = [all_numeric_inputs]

for name, input in inputs.items():
  if input.dtype == tf.float32:
    continue
  lookup = preprocessing.StringLookup(vocabulary=np.unique(air_accident_features[name]))
  one_hot = preprocessing.CategoryEncoding(num_tokens=lookup.vocabulary_size())
  x = lookup(input)
  x = one_hot(x)
  preprocessed_inputs.append(x)

preprocessed_inputs_cat = layers.Concatenate()(preprocessed_inputs)
air_accident_preprocessing = tf.keras.Model(inputs, preprocessed_inputs_cat)
tf.keras.utils.plot_model(model = air_accident_preprocessing , rankdir="LR", dpi=72, show_shapes=True)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


convertimos los datos **panda** en un diccionario de tensores

In [18]:
air_accident_features_dict = {name: np.array(value)
                         for name, value in air_accident_features.items()}

Dividimos el primer ejemplo de entrenamiento y lo pasamos a un modelo de preprocesamiento

In [19]:
features_dict = {name:values[:1] for name, values in air_accident_features_dict.items()}
air_accident_preprocessing(features_dict)

<tf.Tensor: shape=(1, 28), dtype=float32, numpy=
array([[-0.61 ,  0.395, -0.479, -0.497,  0.   ,  0.   ,  1.   ,  0.   ,
         0.   ,  0.   ,  1.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,
         0.   ,  0.   ,  0.   ,  1.   ,  0.   ,  0.   ,  0.   ,  1.   ,
         0.   ,  0.   ,  1.   ,  0.   ]], dtype=float32)>

construimos el modelo
pasamos el diccionario de entidades como x y la etiqueta como y

In [20]:
def air_accident_model(preprocessing_head, inputs):
  body = tf.keras.Sequential([
    layers.Dense(64),
    layers.Dense(1)
  ])
  preprocessed_inputs = preprocessing_head(inputs)
  result = body(preprocessed_inputs)
  model = tf.keras.Model(inputs, result)
  model.compile(loss=tf.losses.BinaryCrossentropy(from_logits=True),
                optimizer=tf.optimizers.Adam())
  return model
air_accident_model = air_accident_model(air_accident_preprocessing, inputs)

air_accident_model.fit(x=air_accident_features_dict, y=air_accident_labels, epochs=10)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7287   
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5935 
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5309 
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4933 
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4682 
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4538 
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4413 
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4351 
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4317 
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4264 


guardamos el modelo

In [37]:
air_accident_model.save('model/test.keras')
reloaded = tf.keras.models.load_model('model/test.keras')

In [38]:
features_dict = {name:values[:1] for name, values in air_accident_features_dict.items()}

before = air_accident_model(features_dict)
after = reloaded(features_dict)
assert (before-after)<1e-3
print(before)
print(after)

tf.Tensor([[-1.903]], shape=(1, 1), dtype=float32)
tf.Tensor([[-1.903]], shape=(1, 1), dtype=float32)


Cargamos un ejemplo

In [41]:
import itertools

def slices(features):
  for i in itertools.count():
    #Para cada característica, tomamos el índice `i`
    example = {name:values[i] for name, values in features.items()}
    yield example
#Ejecutamos primer ejemplo
for example in slices(air_accident_features):
  for name, value in example.items():
    print(f"{name:19s}: {value}")
  break

sex                : male
age                : 22.0
n_siblings_spouses : 1
parch              : 0
fare               : 7.25
class              : Third
deck               : unknown
embark_town        : Southampton
alone              : n


iteramos sobre un **tf.data.Dataset**

In [26]:
features_ds = tf.data.Dataset.from_tensor_slices(air_accident_features_dict)

for example in features_ds:
  for name, value in example.items():
    print(f"{name:19s}: {value}")
  break

sex                : b'male'
age                : 22.0
n_siblings_spouses : 1
parch              : 0
fare               : 7.25
class              : b'Third'
deck               : b'unknown'
embark_town        : b'Southampton'
alone              : b'n'


In [27]:
air_accident_ds = tf.data.Dataset.from_tensor_slices((air_accident_features_dict, air_accident_labels))
air_accident_batches = air_accident_ds.shuffle(len(air_accident_labels)).batch(32)
air_accident_model.fit(air_accident_batches, epochs=5)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step - loss: 0.4245
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 934us/step - loss: 0.4234
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step - loss: 0.4222
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4212
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 964us/step - loss: 0.4214


In [42]:
# Creamos un pasajero de ejemplo manualmente
ejemplo = {
    'sex':                np.array(['male']),
    'age':                np.array([22.0]),
    'n_siblings_spouses': np.array([1]),
    'parch':              np.array([0]),
    'fare':               np.array([7.25]),
    'class':              np.array(['Third']),
    'deck':               np.array(['unknown']),
    'embark_town':        np.array(['Southampton']),
    'alone':              np.array(['n'])
}

# El modelo predice
prediccion = air_accident_model(ejemplo)

# Convertimos el resultado a probabilidad con sigmoid
probabilidad = tf.sigmoid(prediccion).numpy()[0][0]

print(f"Valor crudo del modelo : {prediccion.numpy()[0][0]:.4f}")
print(f"Probabilidad de sobrevivir: {probabilidad:.2%}")
print(f"Predicción final: {'SOBREVIVE ✅' if probabilidad > 0.5 else 'NO SOBREVIVE ❌'}")

Valor crudo del modelo : -1.9033
Probabilidad de sobrevivir: 12.97%
Predicción final: NO SOBREVIVE ❌


leemos los datos CSV del archivo y cree **untf.data.Dataset**